# TrajSurv in PyHealth: in-hospital survival from the first 36 ICU hours

<a href="https://colab.research.google.com/github/sunlabuiuc/PyHealth/blob/master/examples/trajsurv_mimic4_survival.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook reproduces the survival setting of **TrajSurv** (Zeng et al., *TrajSurv: Learning Continuous Latent Trajectories from Electronic Health Records for Trustworthy Survival Prediction*, MLHC 2025, [arXiv:2508.00657](https://arxiv.org/abs/2508.00657); code: [zengsihang/TrajSurv](https://github.com/zengsihang/TrajSurv), MIT License) with PyHealth:

- **Task** (`InHospitalSurvivalMIMIC4`): hourly labs and vitals over the first 36 hours of an ICU stay predict the time to in-hospital death, censored at hospital discharge.
- **Model** (`TrajSurv`): a neural controlled differential equation turns the hourly series into a continuous latent trajectory; a Cox head on the last latent state gives the risk.
- **Baselines**: `SurvivalMDN` (already in PyHealth) and a linear Cox model on a flat summary of the same window.
- **Metrics**: Harrell's C-index, and the IPCW Brier score and cumulative/dynamic AUC at 7, 14 and 28 days (`pyhealth.metrics.survival`).

Part 1 runs end to end on the open **MIMIC-IV Clinical Database Demo** (100 patients, no credentials), as a smoke test. Part 2 runs the same code on the full, credentialed MIMIC-IV; use a GPU runtime (*Runtime → Change runtime type → GPU*).

## Setup

TrajSurv is on PyHealth's master branch and will be in the next PyPI release. Until then, install from GitHub.

In [ ]:
!pip install -q "git+https://github.com/sunlabuiuc/PyHealth"
# after the next PyPI release: !pip install -q pyhealth

In [ ]:
import time

import numpy as np
import torch
from torch import nn

from pyhealth.datasets import MIMIC4EHRDataset, PatientSplit, get_dataloader
from pyhealth.metrics import survival_metrics_fn
from pyhealth.models import SurvivalMDN, TrajSurv
from pyhealth.models.base_model import BaseModel
from pyhealth.models.trajsurv import cox_partial_likelihood_loss
from pyhealth.tasks import InHospitalSurvivalMIMIC4

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## Part 1: the open MIMIC-IV demo

Download the demo tables the task needs from PhysioNet (open access, about 7 MB).

In [ ]:
import os
import urllib.request

DEMO_ROOT = "mimic-iv-demo-2.2"
BASE = "https://physionet.org/files/mimic-iv-demo/2.2/"
TABLES = ["hosp/admissions", "hosp/patients", "hosp/labevents", "hosp/d_labitems",
          "icu/icustays", "icu/chartevents", "icu/outputevents"]
for table in TABLES:
    path = os.path.join(DEMO_ROOT, table + ".csv.gz")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not os.path.exists(path):
        urllib.request.urlretrieve(BASE + table + ".csv.gz", path)
print("downloaded", len(TABLES), "tables")

### Models and helpers

`TrajSurv` reads the hourly `dynamic` series (forward-filled, standardized and masked by the task's processor, with statistics fitted on the training patients only). `SurvivalMDN` and the linear Cox baseline read `summary`: the last value of each hourly variable in the window plus age, sex, height and weight.

TrajSurv and the linear Cox model output a risk score; for SurvivalMDN the risk is minus the restricted mean survival time over the grid. Survival curves feed the Brier score; TrajSurv's curves use a Breslow baseline hazard fitted on the training data.

In [ ]:
HORIZONS = [7.0, 14.0, 28.0]          # days after the first 36 ICU hours
GRID = torch.linspace(0.1, 60.0, 120)  # survival-curve grid, days


class LinearCox(BaseModel):
    """Cox proportional hazards on the flat summary vector (linear baseline)."""

    def __init__(self, dataset, feature_key="summary"):
        super().__init__(dataset)
        self.mode = None
        self.feature_key = feature_key
        self.linear = nn.Linear(dataset[0][feature_key].numel(), 1)

    def forward(self, **batch):
        x = batch[self.feature_key].to(self.device, torch.float32)
        risk = self.linear(x).squeeze(-1)
        duration = batch["duration"].to(self.device).reshape(-1)
        event = batch["event"].to(self.device).reshape(-1)
        return {"risk": risk,
                "loss": cox_partial_likelihood_loss(risk, duration, event),
                "y_true": torch.stack([duration, event], -1)}


@torch.no_grad()
def predict(model, loader):
    model.eval()
    risk, surv, y = [], [], []
    for batch in loader:
        out = model(**batch)
        y.append(out["y_true"].cpu())
        if "risk" in out:
            risk.append(out["risk"].cpu())
        else:  # SurvivalMDN: minus the restricted mean survival time
            risk.append(-torch.trapezoid(out["y_prob"].cpu(), model.time_grid.cpu(), dim=1))
        if "y_prob" in out:
            surv.append(out["y_prob"].cpu())
    y = torch.cat(y).numpy()
    return y[:, 0], y[:, 1], torch.cat(risk).numpy(), torch.cat(surv).numpy() if surv else None


def evaluate(model, loader):
    duration, event, risk, surv = predict(model, loader)
    return survival_metrics_fn(duration, event, risk, survival=surv,
                               time_grid=GRID.numpy(), times=HORIZONS)


def train(model, train_loader, val_loader, lr, epochs):
    """Adam; keeps the epoch with the best validation C-index."""
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best, best_state = -1.0, None
    for _ in range(epochs):
        model.train()
        for batch in train_loader:
            if len(batch["duration"]) < 2:
                # A last batch of one sample: SurvivalMDN's BatchNorm cannot
                # train on it, and a Cox loss on one sample is uninformative.
                continue
            optimizer.zero_grad()
            model(**batch)["loss"].backward()
            optimizer.step()
        c_index = evaluate(model, val_loader)["c_index"]
        if not np.isnan(c_index) and c_index > best:
            best, best_state = c_index, {k: v.detach().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def run(root, cache_dir, epochs, batch_size=64, dev=False):
    dataset = MIMIC4EHRDataset(root=root, cache_dir=cache_dir, dev=dev,
                               tables=["labevents", "chartevents", "outputevents", "admissions"])
    parts = dataset.set_task(InHospitalSurvivalMIMIC4(),
                             split=PatientSplit((0.7, 0.15, 0.15), seed=0))
    for name, part in zip(("train", "val", "test"), parts):
        deaths = int(sum(float(part[i]["event"]) for i in range(len(part))))
        print(f"{name}: {len(part)} ICU stays, {deaths} in-hospital deaths")
    train_dl, val_dl, test_dl = (get_dataloader(p, batch_size=batch_size, shuffle=(i == 0))
                                 for i, p in enumerate(parts))
    models = {
        "TrajSurv": (TrajSurv(parts[0], hidden_dim=64, time_grid=GRID), 3e-4),
        "SurvivalMDN": (SurvivalMDN(parts[0], feature_key="summary", hidden_dim=32,
                                    num_components=5, time_grid=GRID), 1e-3),
        "Linear Cox": (LinearCox(parts[0]), 1e-3),
    }
    results = {}
    for name, (model, lr) in models.items():
        start = time.time()
        train(model, train_dl, val_dl, lr, epochs)
        if isinstance(model, TrajSurv):
            model.fit_baseline_hazard(train_dl)
        results[name] = evaluate(model, test_dl)
        print(f"{name}: trained in {time.time() - start:.0f}s")
    return results


def show(results):
    """Prints a table; the linear Cox model has no survival curves, so no Brier."""
    keys = list(next(iter(results.values())))
    print("model".ljust(12) + "".join(k.rjust(10) for k in keys))
    for name, scores in results.items():
        print(name.ljust(12) + "".join(f"{scores.get(k, float('nan')):10.3f}" for k in keys))

### Train and evaluate on the demo

The demo has about 80 eligible ICU stays and about a dozen deaths, so the test split holds roughly 10 stays: the numbers below only show that the pipeline runs. TrajSurv takes a few minutes per 20 epochs on a CPU.

In [ ]:
demo_results = run(DEMO_ROOT, cache_dir="cache-demo", epochs=20)
show(demo_results)

## Part 2: full MIMIC-IV (credentialed)

MIMIC-IV requires PhysioNet credentialing ([physionet.org/content/mimiciv](https://physionet.org/content/mimiciv/)). Point `MIMIC4_ROOT` at your copy (the folder with `hosp/` and `icu/`), e.g. on a mounted Google Drive, and use a GPU runtime. Building the event table reads `chartevents` (about 30 GB uncompressed) once and caches it in `cache_dir`.

The paper trains with batch size 64, learning rate 3e-4, a 64-dimensional latent state and a 3-layer vector field; those are the defaults here. Set `EPOCHS` to what your budget allows (the authors train until validation early stopping).

In [ ]:
MIMIC4_ROOT = "/content/drive/MyDrive/mimic-iv/2.2"  # change me
EPOCHS = 50

if os.path.isdir(MIMIC4_ROOT):
    full_results = run(MIMIC4_ROOT, cache_dir="cache-mimic4", epochs=EPOCHS)
    show(full_results)
else:
    print("Set MIMIC4_ROOT to your MIMIC-IV folder to run Part 2.")

## Comparison with the paper

TrajSurv's reported test results (Table 2 of the paper, mean over runs):

| Dataset | Model | C-index | Brier | AUC |
|---|---|---|---|---|
| MIMIC-III | TrajSurv | 0.803 ± 0.011 | 0.056 | 0.790 |
| MIMIC-III | best baseline (RSF) | 0.784 | | |
| eICU | TrajSurv | 0.823 ± 0.006 | 0.044 | 0.803 |
| eICU | best baseline (DDH) | 0.818 | | |

What differs here, so numbers are comparable in trend rather than identical:

- **Data**: MIMIC-IV instead of MIMIC-III (MetaVision stays) and eICU. The cohort is the YAIB base cohort (adults with enough ICU measurements) restricted to the first ICU stay of 36 hours or more per admission, alive and in hospital at hour 36.
- **Features**: the 48 hourly YAIB variables (labs, vitals, urine output), close to the paper's 53, forward-filled and standardized by the task's processor, plus the fraction of hours observed so far per variable (the paper uses cumulative observation counts).
- **Contrastive term**: off. The paper aligns latent states with the SOFA score and its trend; PyHealth's task has no SOFA series yet. `TrajSurv(..., alignment_key=...)` takes any per-hour alignment target (e.g. SOFA) when one is added to the samples.
- **Solver**: natural cubic spline path and fixed-step RK4 in pure PyTorch, instead of `torchcde`'s adaptive solver.
- **Horizons**: the Brier score and AUC are reported at 7, 14 and 28 days after hour 36; the paper's horizon choice may differ.

## What to check (for the TrajSurv authors)

1. The cohort and censoring definition in `pyhealth/tasks/survival_mimic4.py` against your MIMIC-III cohort.
2. The model in `pyhealth/models/trajsurv.py`: input channels, vector field and Cox head against `ncde/model.py`.
3. Whether the Rank-N-Contrast reimplementation (`rank_n_contrast_loss`, time-aware via the time difference added to the label distance) matches the variant you used.
4. Whether the evaluation horizons and metric definitions (`pyhealth/metrics/survival.py`) match Table 2.